In [1]:
ls $(python -c "import pyspark, os; print(os.path.dirname(pyspark.__file__))")/jars | grep -E "scala-library|spark-core"

scala-library-2.13.18.jar
spark-core_2.13-4.2.0.jar


In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from dotenv import load_dotenv

load_dotenv()

CLIENT_ID = os.environ["GCN_CLIENT_ID"]
CLIENT_SECRET = os.environ["GCN_CLIENT_SECRET"]

TOPICS = ",".join(
    [
        "gcn.notices.boom.alert",
        "gcn.circulars",
        "gcn.notices.dsa110.frb",
        "gcn.heartbeat",
        "igwn.gwalert",
        "gcn.notices.maxi.gsc.known",
        "gcn.notices.maxi.gsc.unknown",
        "gcn.notices.superk.sn_alert",
        "gcn.notices.swift.bat.guano",
    ]
)

spark = (
    SparkSession.builder.config(
        "spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"
    )
    .appName("gcn_example")
    .getOrCreate()
)

kafka = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "kafka.gcn.nasa.gov:9092")
    .option("kafka.security.protocol", "SASL_SSL")
    .option("kafka.sasl.mechanism", "OAUTHBEARER")
    .option(
        # kafka-clients 3.x path. If 4.2.0 bundles kafka-clients 4.x, drop
        # ".secured" — verify with the unzip check against the resolved JAR.
        "kafka.sasl.login.callback.handler.class",
        "org.apache.kafka.common.security.oauthbearer.secured.OAuthBearerLoginCallbackHandler",
    )
    .option(
        "kafka.sasl.jaas.config",
        "org.apache.kafka.common.security.oauthbearer.OAuthBearerLoginModule required "
        f'clientId="{CLIENT_ID}" clientSecret="{CLIENT_SECRET}";',
    )
    .option(
        "kafka.sasl.oauthbearer.token.endpoint.url",
        "https://auth.gcn.nasa.gov/oauth2/token",
    )
    .option("subscribe", TOPICS)
    .option("startingOffsets", "earliest")
    .option("maxOffsetsPerTrigger", 500)
    .option("failOnDataLoss", "false")
    .load()
)

decoded = kafka.select(
    col("topic"),
    col("timestamp"),
    col("value").cast("string").alias("payload"),
)

query = decoded.writeStream.format("console").option("truncate", False).start()
query.awaitTermination(120)

print(f"Batches: {len(query.recentProgress)}")
print(f"Rows: {sum(p['sources'][0]['numInputRows'] for p in query.recentProgress)}")

query.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/05 16:50:36 WARN Utils: Your hostname, Harrys-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.199 instead (on interface en0)
26/09/05 16:50:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/opt/homebrew/Caskroom/miniforge/base/envs/gcn/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/harrywatson/.ivy2.5.2/cache
The jars for the packages stored in: /Users/harrywatson/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-78110f8d-b97d-4fc4-b240-e35fd5d62231;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apach

-------------------------------------------
Batch: 0
-------------------------------------------
+-------------+-----------------------+------------------------------------------------------------------------------------------------------------------------------------------+
|topic        |timestamp              |payload                                                                                                                                   |
+-------------+-----------------------+------------------------------------------------------------------------------------------------------------------------------------------+
|gcn.heartbeat|2026-08-07 05:49:29.161|{"$schema": "https://gcn.nasa.gov/docs/schema/v4.1.0/gcn/notices/core/Alert.schema.json", "alert_datetime": "2026-08-06T19:49:29.161917Z"}|
|gcn.heartbeat|2026-08-07 05:49:30.163|{"$schema": "https://gcn.nasa.gov/docs/schema/v4.1.0/gcn/notices/core/Alert.schema.json", "alert_datetime": "2026-08-06T19:49:30.163190Z"}|
|gcn.hea

-------------------------------------------
Batch: 1
-------------------------------------------
+-------------+-----------------------+------------------------------------------------------------------------------------------------------------------------------------------+
|topic        |timestamp              |payload                                                                                                                                   |
+-------------+-----------------------+------------------------------------------------------------------------------------------------------------------------------------------+
|gcn.heartbeat|2026-08-07 05:57:48.751|{"$schema": "https://gcn.nasa.gov/docs/schema/v4.1.0/gcn/notices/core/Alert.schema.json", "alert_datetime": "2026-08-06T19:57:48.751938Z"}|
|gcn.heartbeat|2026-08-07 05:57:49.753|{"$schema": "https://gcn.nasa.gov/docs/schema/v4.1.0/gcn/notices/core/Alert.schema.json", "alert_datetime": "2026-08-06T19:57:49.753214Z"}|
|gcn.hea

-------------------------------------------
Batch: 2
-------------------------------------------
+-------------+-----------------------+------------------------------------------------------------------------------------------------------------------------------------------+
|topic        |timestamp              |payload                                                                                                                                   |
+-------------+-----------------------+------------------------------------------------------------------------------------------------------------------------------------------+
|gcn.heartbeat|2026-08-07 06:06:08.354|{"$schema": "https://gcn.nasa.gov/docs/schema/v4.1.0/gcn/notices/core/Alert.schema.json", "alert_datetime": "2026-08-06T20:06:08.354754Z"}|
|gcn.heartbeat|2026-08-07 06:06:09.355|{"$schema": "https://gcn.nasa.gov/docs/schema/v4.1.0/gcn/notices/core/Alert.schema.json", "alert_datetime": "2026-08-06T20:06:09.355039Z"}|
|gcn.hea

-------------------------------------------
Batch: 3
-------------------------------------------
+-------------+-----------------------+------------------------------------------------------------------------------------------------------------------------------------------+
|topic        |timestamp              |payload                                                                                                                                   |
+-------------+-----------------------+------------------------------------------------------------------------------------------------------------------------------------------+
|gcn.heartbeat|2026-08-07 06:14:27.959|{"$schema": "https://gcn.nasa.gov/docs/schema/v4.1.0/gcn/notices/core/Alert.schema.json", "alert_datetime": "2026-08-06T20:14:27.959615Z"}|
|gcn.heartbeat|2026-08-07 06:14:28.96 |{"$schema": "https://gcn.nasa.gov/docs/schema/v4.1.0/gcn/notices/core/Alert.schema.json", "alert_datetime": "2026-08-06T20:14:28.960074Z"}|
|gcn.hea

ERROR:root:Exception while sending command.
Traceback (most recent call last):
  File "/opt/homebrew/Caskroom/miniforge/base/envs/gcn/lib/python3.14/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ~~~~~~~~~~~~~~~~~~~~^^
RuntimeError: reentrant call inside <_io.BufferedReader name=90>

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/homebrew/Caskroom/miniforge/base/envs/gcn/lib/python3.14/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/opt/homebrew/Caskroom/miniforge/base/envs/gcn/lib/python3.14/site-packages/py4j/clientserver.py", line 566, in send_command
    raise Py4JNetworkError(
        "Error while sending or receiving", e, proto.ERROR_ON_RECEIVE)
py4j.protocol.Py4JNetworkError: Error while sending or receiving
ERROR:root:KeyboardInterrupt while 

KeyboardInterrupt: 

-------------------------------------------
Batch: 4
-------------------------------------------
+-------------+-----------------------+------------------------------------------------------------------------------------------------------------------------------------------+
|topic        |timestamp              |payload                                                                                                                                   |
+-------------+-----------------------+------------------------------------------------------------------------------------------------------------------------------------------+
|gcn.heartbeat|2026-08-07 06:22:47.568|{"$schema": "https://gcn.nasa.gov/docs/schema/v4.1.0/gcn/notices/core/Alert.schema.json", "alert_datetime": "2026-08-06T20:22:47.568431Z"}|
|gcn.heartbeat|2026-08-07 06:22:48.569|{"$schema": "https://gcn.nasa.gov/docs/schema/v4.1.0/gcn/notices/core/Alert.schema.json", "alert_datetime": "2026-08-06T20:22:48.569725Z"}|
|gcn.hea